# 04. Классификация направления BTC (v2)

Построение ML-модели для предсказания бинарного направления движения цены BTC через 7 дней.

**Улучшения по сравнению с предыдущей версией:**
1. Расширенный набор признаков (FNG, EFFR, RSI, MACD, Bollinger, Hurst, корреляция BTC-SPX)
2. Чистая целевая переменная `up_7d_ahead`
3. Три модели для сравнения: XGBoost, Random Forest, BaggingClassifier
4. Полные метрики классификации: Accuracy, Precision, Recall, F1, AUC-ROC
5. Анализ важности признаков (permutation importance)

**Методология валидации:** Combinatorial Purged Cross-Validation (CPCV) с embargo.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, log_loss,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve)
from sklearn.inspection import permutation_importance

from src.common.split import cpcv_generator, purge, embargo

np.random.seed(42)
plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True})

In [ ]:
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print("XGBoost доступен")
except ImportError:
    HAS_XGB = False
    print("XGBoost не установлен. Установите: pip install xgboost")
    print("Будет использован GradientBoostingClassifier как замена.")
    from sklearn.ensemble import GradientBoostingClassifier

## 1. Загрузка данных и подготовка

Целевая переменная: $y_t = \mathbb{1}(P_{t+7} > P_t)$ — бинарная метка «цена через 7 дней выше текущей».

Бейзлайн «всегда вверх» даёт accuracy ≈ 54.9% (из `02_Baselines.ipynb`), т.к. BTC имеет положительный drift.

In [ ]:
# Загрузка признаков и таргетов
features = pd.read_csv("../data/features/features_v1.csv", parse_dates=["date"]).set_index("date")
targets  = pd.read_csv("../data/targets/targets_v1.csv",   parse_dates=["date"]).set_index("date")

# Объединяем по дате
data = features.join(targets, how="inner").dropna()
print(f"Объединённый датасет: {data.shape[0]} строк, {data.shape[1]} столбцов")

# Выбираем признаки (исключаем сырые цены — нестационарны, см. 03_Distribution_Analysis)
price_cols = ["btc_close", "btc_ma7", "btc_ma20", "btc_ma60", "spx_close", "spx_ma20"]
target_cols = [c for c in data.columns if c.startswith("ret_") or c.startswith("up_")]
feature_cols = [c for c in data.columns if c not in price_cols + target_cols]

print(f"Признаки ({len(feature_cols)}): {feature_cols}")
print(f"\nЦелевая переменная: up_7d_ahead")
print(f"Баланс классов: {data['up_7d_ahead'].value_counts().to_dict()}")
print(f"Доля класса 1 (up): {data['up_7d_ahead'].mean():.3f}")

X = data[feature_cols]
y = data["up_7d_ahead"]

In [ ]:
# Матрица корреляций признаков
plt.figure(figsize=(12, 10))
corr = X.corr()
im = plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, shrink=0.8)
plt.xticks(range(len(feature_cols)), feature_cols, rotation=90, fontsize=8)
plt.yticks(range(len(feature_cols)), feature_cols, fontsize=8)
plt.title("Матрица корреляций признаков")
plt.tight_layout()
plt.show()

# Высокие корреляции (|r| > 0.8)
high_corr = []
for i in range(len(feature_cols)):
    for j in range(i+1, len(feature_cols)):
        if abs(corr.iloc[i, j]) > 0.8:
            high_corr.append((feature_cols[i], feature_cols[j], corr.iloc[i, j]))
if high_corr:
    print("Пары признаков с |r| > 0.8:")
    for a, b, r in high_corr:
        print(f"  {a} ↔ {b}: r = {r:.3f}")

## 2. CPCV: Combinatorial Purged Cross-Validation

Стандартный k-fold не подходит для временных рядов из-за временной зависимости.

CPCV (de Prado, 2018) решает эту проблему:
- Делит временной ряд на $n$ последовательных групп
- Перебирает все $C(n,k)$ комбинаций из $k$ тестовых групп → $C(n,k)$ симуляций
- **Purging:** удаляет из обучающей выборки наблюдения, чьи метки перекрываются с тестовой
- **Embargo:** дополнительно удаляет буферную зону после теста, чтобы предотвратить утечку через автокорреляцию

Для $n=6, k=2$: $C(6,2) = 15$ симуляций, $15 \cdot 2/6 = 5$ путей.

In [ ]:
# Настройка CPCV
n_groups, k_test = 6, 2
horizon = 7  # горизонт предсказания

t_span = X.shape[0]
is_test, paths, path_folds = cpcv_generator(t_span, n_groups, k_test)
n_sim = is_test.shape[1]
n_paths = paths.shape[1]

print(f"Наблюдений: {t_span}")
print(f"Групп: {n_groups}, тестовых: {k_test}")
print(f"Симуляций: {n_sim}")
print(f"Путей: {n_paths}")

# t1 — Series: для каждого наблюдения отмечает конец интервала метки
# (т.е. момент, до которого используется будущая информация)
times = X.index
t1 = pd.Series(index=times, dtype="datetime64[ns]")
for i, dt in enumerate(times):
    end_idx = min(i + horizon, len(times) - 1)
    t1.iloc[i] = times[end_idx]

## 3. Обучение моделей

Сравниваем три модели:

| Модель | Описание | Обоснование |
|--------|----------|-------------|
| **XGBoost** (или GradientBoosting) | Градиентный бустинг | Последовательно строит деревья, исправляя ошибки; встроенная L1/L2-регуляризация |
| **Random Forest** | Бэггинг деревьев | Параллельные деревья; робастен к переобучению |
| **BaggingClassifier** | Бэггинг (как в v1, исправленный) | Для сравнения с предыдущей версией |

Для каждой из $C(6,2) = 15$ CPCV-симуляций обучаем модель и собираем метрики.

In [ ]:
# Определение моделей
pos_weight = (y == 0).sum() / (y == 1).sum()

if HAS_XGB:
    boost_model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        random_state=42, eval_metric="logloss", verbosity=0,
    )
else:
    boost_model = GradientBoostingClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=42,
    )

models = {
    "XGBoost": boost_model,
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=7, min_samples_leaf=20,
        class_weight="balanced", random_state=42, n_jobs=-1,
    ),
    "Bagging (v1 fix)": BaggingClassifier(
        estimator=DecisionTreeClassifier(
            criterion="entropy", max_features="sqrt",
            class_weight="balanced", min_weight_fraction_leaf=0.05,
        ),
        n_estimators=50, max_features=0.9, random_state=42, n_jobs=-1,
    ),
}

print("Модели:")
for name in models:
    print(f"  - {name}")

In [ ]:
from sklearn.base import clone
from tqdm.auto import tqdm

def train_evaluate_cpcv(model, X, y, t1, is_test, pct_embargo=0.01):
    """Обучает модель на каждой CPCV-симуляции, возвращает метрики по фолдам."""
    n_sim = is_test.shape[1]
    all_metrics = []
    all_y_true, all_y_pred, all_y_proba = [], [], []

    for k in tqdm(range(n_sim), desc=model.__class__.__name__):
        test_idx = is_test[:, k]
        test_times = t1.loc[test_idx]
        test_times_emb = embargo(test_times, t1, pct_embargo=pct_embargo)
        train_times = purge(t1, test_times_emb)

        X_test  = X.loc[test_times.index]
        y_test  = y.loc[X_test.index]
        X_train = X.loc[train_times.index]
        y_train = y.loc[X_train.index]

        clf = clone(model)
        clf.fit(X_train, y_train)

        y_pred  = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]

        all_y_true.append(y_test)
        all_y_pred.append(pd.Series(y_pred, index=y_test.index))
        all_y_proba.append(pd.Series(y_proba, index=y_test.index))

        all_metrics.append({
            "sim": k,
            "accuracy":  accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall":    recall_score(y_test, y_pred, zero_division=0),
            "f1":        f1_score(y_test, y_pred, zero_division=0),
            "auc_roc":   roc_auc_score(y_test, y_proba),
            "logloss":   log_loss(y_test, y_proba),
            "n_train":   len(X_train),
            "n_test":    len(X_test),
        })

    return pd.DataFrame(all_metrics), all_y_true, all_y_pred, all_y_proba

# Обучение всех моделей
results = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Обучение: {name}")
    metrics_df, yt, yp, ypr = train_evaluate_cpcv(model, X, y, t1, is_test)
    results[name] = {"metrics": metrics_df, "y_true": yt, "y_pred": yp, "y_proba": ypr}
    print(f"Средняя accuracy: {metrics_df['accuracy'].mean():.4f} ± {metrics_df['accuracy'].std():.4f}")

## 4. Метрики классификации

Для каждой модели вычислены по всем 15 CPCV-симуляциям:
- **Accuracy** — доля правильных предсказаний
- **Precision** — точность: $\frac{TP}{TP+FP}$ — из предсказанных «вверх», сколько действительно выросли
- **Recall** — полнота: $\frac{TP}{TP+FN}$ — из действительно выросших, сколько предсказали
- **F1** — гармоническое среднее precision и recall: $\frac{2 \cdot P \cdot R}{P + R}$
- **AUC-ROC** — площадь под ROC-кривой (качество ранжирования)
- **Log-loss** — логарифмическая функция потерь: $-\frac{1}{n}\sum[y\ln p + (1-y)\ln(1-p)]$

In [ ]:
# Сводная таблица: mean ± std по CPCV-симуляциям
metric_cols = ["accuracy", "precision", "recall", "f1", "auc_roc", "logloss"]

summary_rows = []
for name, res in results.items():
    m = res["metrics"]
    row = {"Модель": name}
    for col in metric_cols:
        row[col] = f"{m[col].mean():.4f} ± {m[col].std():.4f}"
    summary_rows.append(row)

# Добавляем бейзлайн «всегда вверх»
baseline_acc = y.mean()
summary_rows.append({
    "Модель": "Бейзлайн (всегда UP)",
    "accuracy": f"{baseline_acc:.4f}",
    "precision": f"{baseline_acc:.4f}",
    "recall": "1.0000",
    "f1": f"{2*baseline_acc/(1+baseline_acc):.4f}",
    "auc_roc": "0.5000",
    "logloss": f"{-np.log(baseline_acc)*baseline_acc - np.log(1-baseline_acc)*(1-baseline_acc):.4f}",
})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# Boxplot метрик по CPCV-симуляциям
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for i, col in enumerate(metric_cols):
    ax = axes[i // 3, i % 3]
    data_plot = [results[name]["metrics"][col].values for name in results]
    bp = ax.boxplot(data_plot, labels=list(results.keys()), patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#2196F3", "#4CAF50", "#FF9800"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    if col == "accuracy":
        ax.axhline(baseline_acc, color="red", ls="--", lw=1.5, label="Бейзлайн (always UP)")
        ax.legend(fontsize=8)
    ax.set_title(col.upper())
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Метрики по 15 CPCV-симуляциям", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix (агрегированная по всем симуляциям) для лучшей модели
best_name = max(results, key=lambda n: results[n]["metrics"]["accuracy"].mean())
print(f"Лучшая модель по средней accuracy: {best_name}")

best_res = results[best_name]
y_true_all = pd.concat(best_res["y_true"])
y_pred_all = pd.concat(best_res["y_pred"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_true_all, y_pred_all)
ConfusionMatrixDisplay(cm, display_labels=["DOWN", "UP"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title(f"Confusion Matrix ({best_name}, агрег.)")

# ROC-кривые по симуляциям
for k in range(len(best_res["y_true"])):
    fpr, tpr, _ = roc_curve(best_res["y_true"][k], best_res["y_proba"][k])
    axes[1].plot(fpr, tpr, alpha=0.3, lw=1)

# Средняя ROC
y_proba_all = pd.concat(best_res["y_proba"])
fpr_m, tpr_m, _ = roc_curve(y_true_all, y_proba_all)
auc_m = roc_auc_score(y_true_all, y_proba_all)
axes[1].plot(fpr_m, tpr_m, "b-", lw=2, label=f"Средняя (AUC={auc_m:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Случайный (AUC=0.5)")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title(f"ROC-кривые ({best_name})")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Анализ важности признаков

### Permutation Importance
Важность признака $j$ — снижение accuracy модели при случайном перемешивании значений этого признака:

$$PI_j = \text{score}(y, f(X)) - \text{score}(y, f(X_{\text{shuffled}_j}))$$

Метод модель-агностичный и измеряет реальный вклад признака в предсказание, а не просто его использование в дереве (в отличие от Gini importance).

In [ ]:
# Permutation importance для лучшей модели
# Обучаем на последнем CPCV-фолде для наглядности
last_sim = n_sim - 1
test_idx = is_test[:, last_sim]
test_times = t1.loc[test_idx]
test_times_emb = embargo(test_times, t1, pct_embargo=0.01)
train_times = purge(t1, test_times_emb)

X_train_pi = X.loc[train_times.index]
y_train_pi = y.loc[X_train_pi.index]
X_test_pi  = X.loc[test_times.index]
y_test_pi  = y.loc[X_test_pi.index]

best_model_copy = clone(models[best_name])
best_model_copy.fit(X_train_pi, y_train_pi)

perm_imp = permutation_importance(
    best_model_copy, X_test_pi, y_test_pi,
    n_repeats=20, random_state=42, scoring="accuracy",
)

# Сортировка по средней важности
imp_df = pd.DataFrame({
    "Признак": feature_cols,
    "Важность (mean)": perm_imp.importances_mean,
    "Важность (std)":  perm_imp.importances_std,
}).sort_values("Важность (mean)", ascending=False)

display(imp_df)

# Визуализация
fig, ax = plt.subplots(figsize=(10, 8))
imp_sorted = imp_df.sort_values("Важность (mean)", ascending=True)
colors = ["#4CAF50" if v > 0 else "#f44336" for v in imp_sorted["Важность (mean)"]]
ax.barh(imp_sorted["Признак"], imp_sorted["Важность (mean)"],
        xerr=imp_sorted["Важность (std)"], color=colors, alpha=0.7)
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("Снижение accuracy при перемешивании")
ax.set_title(f"Permutation Importance ({best_name})")
plt.tight_layout()
plt.show()

## 6. Выводы

### Сравнение с бейзлайнами
- **Бейзлайн «всегда UP»:** accuracy ≈ 54.9% (7d горизонт)
- **Бейзлайн MA-60:** accuracy ≈ 55.0%

### Методологические улучшения v2 vs v1
1. Расширенный набор признаков (FNG, EFFR, RSI, MACD, Bollinger, Hurst, корреляция BTC-SPX)
2. Чистая целевая переменная вместо 4-квадрантной условной метки
3. Полные метрики классификации (accuracy, precision, recall, F1, AUC-ROC, log-loss)
4. Три модели для сравнения
5. Анализ важности признаков

In [ ]:
# Финальная таблица: лучшая модель vs бейзлайны
best_metrics = results[best_name]["metrics"]

print(f"Лучшая модель: {best_name}")
print(f"{'='*50}")
print(f"Accuracy:   {best_metrics['accuracy'].mean():.4f} ± {best_metrics['accuracy'].std():.4f}  (бейзлайн: {baseline_acc:.4f})")
print(f"Precision:  {best_metrics['precision'].mean():.4f} ± {best_metrics['precision'].std():.4f}")
print(f"Recall:     {best_metrics['recall'].mean():.4f} ± {best_metrics['recall'].std():.4f}")
print(f"F1:         {best_metrics['f1'].mean():.4f} ± {best_metrics['f1'].std():.4f}")
print(f"AUC-ROC:    {best_metrics['auc_roc'].mean():.4f} ± {best_metrics['auc_roc'].std():.4f}  (бейзлайн: 0.5000)")
print(f"Log-loss:   {best_metrics['logloss'].mean():.4f} ± {best_metrics['logloss'].std():.4f}")

improvement = best_metrics["accuracy"].mean() - baseline_acc
print(f"\nУлучшение accuracy над бейзлайном: +{improvement*100:.2f} п.п.")